- DINOv2 models that include "reg" in their name use 4 register tokens.

- For models without register tokens (num_register_tokens = 0):
[ CLS token | Patch token 1 | Patch token 2 | ... | Patch token N ]

- For models with register tokens (num_register_tokens > 0):
[ CLS token | Register token 1 | ... | Register token K | Patch token 1 | Patch token 2 | ... | Patch token N ]
(where K is num_register_tokens)

In [ ]:
# Install required libraries
!uv pip install -q xformers torch torchvision albumentationsx grad-cam

In [ ]:
import os
import sys
import logging
from dataclasses import dataclass, field
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import albumentations as A
import matplotlib.pyplot as plt
from albumentations.pytorch import ToTensorV2
from sklearn.preprocessing import LabelEncoder
from itertools import cycle, islice
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

In [ ]:
# Mount Google Drive and change directory
print("Connecting to Google Drive...")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    os.chdir("/content/drive/MyDrive/")
except Exception as e:
    print(f"Could not mount drive or change directory: {e}")

print(f"Environment setup complete. Current directory: {os.getcwd()}")

Connecting to Google Drive...
Mounted at /content/drive
Environment setup complete. Current directory: /content/drive/MyDrive


In [ ]:
# ----------------------------
# Hardware Check
# ----------------------------

def check_hardware() -> torch.device:
    if not torch.cuda.is_available():
        logger.warning("No GPU detected. Using CPU.")
        return torch.device("cpu")
    device_index = torch.cuda.current_device()
    device = torch.device(f"cuda:{device_index}")
    name = torch.cuda.get_device_name()
    mem = torch.cuda.get_device_properties().total_memory / 1024**3
    print(f"Training on GPU: {name} with {mem:.1f} GB")
    return device

import multiprocessing
# sklearn and multiprocessing
cores = multiprocessing.cpu_count()
print(f"Number of CPU cores: {cores}")
check_hardware()

Number of CPU cores: 8
Training on GPU: Tesla T4 with 14.7 GB


device(type='cuda', index=0)

In [ ]:
#@title Workflow for DINOv2 with Grad-CAM

# ==============================================================================
# PART 1: DEFINITIONS AND MODEL LOADING
# ==============================================================================
print("\n--- [1/3] Loading all definitions and models ---")

# --- Logger Setup ---
LOGGER_NAME = "dino_gradcam_final"
logger = logging.getLogger(LOGGER_NAME)
def setup_logging(log_level: str = "INFO") -> None:
    logger.setLevel(getattr(logging, log_level.upper(), logging.INFO))

    if not logger.handlers:
        ch = logging.StreamHandler()
        ch.setFormatter(logging.Formatter("%(asctime)s %(levelname)s: %(message)s"))
        logger.addHandler(ch)
    logger.propagate = False
setup_logging()

# --- Configuration and definitions ---
@dataclass
class Config:
    model_name: str = "dinov2_vitb14_reg"
    dropout_rate: float = 0.5
    data_dir: Path = Path("./batdrive/OC/P6/P6_data/Images/")
    data_csv: Path = Path("./batdrive/OC/P6/P6_data/df_cleaned.csv")
    output_dir: Path = Path("./batdrive/models/outputs/dino/")
    required_columns: list = field(default_factory=lambda: ["image", "level_1"])

def get_transforms() -> A.Compose:
    return A.Compose([A.SmallestMaxSize(max_size=256, interpolation=cv2.INTER_AREA),
                      A.CenterCrop(224, 224), A.Normalize(mean=(0.485, 0.456, 0.406),
                      std=(0.229, 0.224, 0.225)),
                      ToTensorV2()])

# --- DINOv2Classifier ---
class DINOv2Classifier(nn.Module):
    def __init__(self, num_classes: int, dropout_rate: float, model_name: str):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', model_name, verbose=False)
        for p in self.backbone.parameters(): p.requires_grad = True
        # Expose le nombre de registres
        self.num_register_tokens = self.backbone.num_register_tokens
        self.head = nn.Sequential(nn.Dropout(dropout_rate), nn.Linear(self.backbone.embed_dim, num_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.head(features)

# --- Main Loading Logic ---
try:
    cfg = Config()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")

    misclassified_csv_path = cfg.output_dir / "misclassified_samples.csv"
    df_mis = pd.read_csv(misclassified_csv_path, index_col='Image File')
    logger.info(f"Loaded {len(df_mis)} misclassified samples.")

    df_all = pd.read_csv(cfg.data_csv, usecols=cfg.required_columns)
    le = LabelEncoder().fit(df_all["level_1"])
    class_names = le.classes_.tolist()
    num_classes = len(class_names)

    model_weights_path = cfg.output_dir / "best_head.pth"
    dinov2_model = DINOv2Classifier(num_classes=num_classes, dropout_rate=cfg.dropout_rate, model_name=cfg.model_name)
    dinov2_model.head.load_state_dict(torch.load(model_weights_path, map_location=device))
    dinov2_model.to(device).eval()
    logger.info("DINOv2 model loaded successfully.")

except Exception as e:
    logger.error(f"An error occurred during data/model loading: {e}")
    dinov2_model = None

print("All definitions are loaded.")

# ==============================================================================
# PART 2: INITIALIZE GRAD-CAM AND ITERATOR
# ==============================================================================
print("\n--- [2/3] Initializing Grad-CAM and Iterator ---")

cam = None
misclassified_iterator = None

if dinov2_model is not None:
    target_layers = [dinov2_model.backbone.blocks[-1].norm1]

    def create_dino_reshape_transform(model: nn.Module):
        """
        reshape_transforms'adapte aux modèles DINOv2 avec ou sans tokens de registre
        """
        # Le décalage est 1 (pour le token CLS) + le nombre de tokens de registre
        start_index = 1 + model.num_register_tokens
        logger.info(f"Reshape transform created. CLS token: 1, Register tokens: {model.num_register_tokens}. Slicing from index {start_index}.")

        def reshape_transform(tensor: torch.Tensor) -> torch.Tensor:
            result = tensor[:, start_index:, :]
            # grille 2D
            height = width = int(result.shape[1]**0.5)
            result = result.reshape(result.shape[0], height, width, result.shape[2])
            # change les dimensions pour correspondre au format attendu par Grad-CAM (B, C, H, W)
            return result.permute(0, 3, 1, 2)

        return reshape_transform

    dynamic_reshape_transform = create_dino_reshape_transform(dinov2_model)

    cam = GradCAM(model=dinov2_model, target_layers=target_layers, reshape_transform=dynamic_reshape_transform)

    misclassified_iterator = cycle(df_mis.index.tolist())

    print("Grad-CAM and misclassified iterator are ready.")
else:
    print("⚠️ Skipping Grad-CAM initialization due to previous errors.")

2025-11-21 17:07:20,896 INFO: Using device: cuda



--- [1/3] Loading all definitions and models ---


2025-11-21 17:07:21,582 INFO: Loaded 20 misclassified samples.


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_reg4_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_reg4_pretrain.pth


100%|██████████| 330M/330M [00:01<00:00, 304MB/s]
2025-11-21 17:07:26,870 INFO: DINOv2 model loaded successfully.
2025-11-21 17:07:26,871 INFO: Reshape transform created. CLS token: 1, Register tokens: 4. Slicing from index 5.


All definitions are loaded.

--- [2/3] Initializing Grad-CAM and Iterator ---
Grad-CAM and misclassified iterator are ready.


In [ ]:
#@title 3. Generate & Plot all misclassified explanations

print("\n--- [3/3] Generating visualizations ---")


def overlay_cam_on_image(img_rgb: np.ndarray, heatmap: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    """Resizes a heatmap to match the image and overlays it."""
    heatmap = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_MAGMA)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    overlayed_img = cv2.addWeighted(np.uint8(img_rgb), 1 - alpha, heatmap, alpha, 0)
    return overlayed_img

if 'df_mis' in locals() and not df_mis.empty and cam is not None:

    total_samples = len(df_mis)
    print(f"\n--- Generating Grad-CAM explanations for all {total_samples} misclassified samples ---")

    # Loop directly over the DataFrame index
    for i, fname in enumerate(df_mis.index):
        print(f"Processing sample {i+1}/{total_samples}: {fname}")
        row = df_mis.loc[fname]
        true_lbl, pred_lbl = row["True class"], row["Predicted class"]
        t_i, p_i = class_names.index(true_lbl), class_names.index(pred_lbl)

        img_bgr = cv2.imread(str(cfg.data_dir / fname))
        if img_bgr is None:
            logger.warning(f"Couldn’t read {fname}, skipping")
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        input_tensor = get_transforms()(image=img_rgb)["image"].unsqueeze(0).to(device)

        targets_true = [ClassifierOutputTarget(t_i)]
        targets_pred = [ClassifierOutputTarget(p_i)]

        grayscale_cam_true = cam(input_tensor=input_tensor, targets=targets_true, eigen_smooth=True)[0, :]
        grayscale_cam_pred = cam(input_tensor=input_tensor, targets=targets_pred, eigen_smooth=True)[0, :]

        vis_true = overlay_cam_on_image(img_rgb, grayscale_cam_true)
        vis_pred = overlay_cam_on_image(img_rgb, grayscale_cam_pred)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle(f"File: {fname}", fontsize=14)
        axes[0].imshow(img_rgb); axes[0].set_title("Original"); axes[0].axis("off")
        axes[1].imshow(vis_true); axes[1].set_title(f"CAM for True: {true_lbl}"); axes[1].axis("off")
        axes[2].imshow(vis_pred); axes[2].set_title(f"CAM for Pred: {pred_lbl}"); axes[2].axis("off")
        plt.tight_layout(rect=[0, 0.03, 1, 0.95]); plt.show()

    print(f"--- Done plotting all {total_samples} samples ---")
else:
    print("⚠️ Plotting skipped due to setup errors or empty misclassified list.")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
#@title cleanup

del cam
print("Cleaned up GradCAM object.")

Cleaned up GradCAM object.
